# 02 — Library Census

Point acidcat at a **whole directory**, profile every file's container, and let
the aggregate talk: format distribution, chunk-frequency census, and the oddballs
that don't fit (shiny candidates).

Default target is the synthetic specimen set. Point `LIBRARY` at `~/sample_packs`
(1,286 real WAVs) for a census with teeth.

In [ ]:
import acidcat, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from collections import Counter
sns.set_theme(style='whitegrid')

PLAYGROUND = Path('~/acidcat-playground').expanduser()
LIBRARY = PLAYGROUND / 'specimens'      # or: Path('~/sample_packs').expanduser()

AUDIO_EXT = {'.wav','.aif','.aiff','.aifc','.flac','.mp3','.m4a','.ogg','.opus',
             '.mid','.rmid','.wt','.multisample','.fxp','.vital','.bwpreset'}
files = [p for p in LIBRARY.rglob('*') if p.is_file() and p.suffix.lower() in AUDIO_EXT]
print(f'{len(files)} candidate files under {LIBRARY}')

## 1. Walk every file

One row per file: format, chunk ids, size, warning count.

In [ ]:
rows = []
for p in files:
    rec = {'file': p.name, 'ext': p.suffix.lower(), 'size': p.stat().st_size,
           'format': None, 'n_chunks': 0, 'chunk_ids': [], 'warnings': 0}
    try:
        fmt, chunks, fwarns = acidcat.walk_file(str(p))
        rec['format'] = fmt
        rec['n_chunks'] = len(chunks)
        rec['chunk_ids'] = [c['id'] for c in chunks]
        rec['warnings'] = sum(len(c.get('warnings') or []) for c in chunks) + len(fwarns)
    except Exception as e:
        rec['format'] = f'<unsupported: {type(e).__name__}>'
    rows.append(rec)
df = pd.DataFrame(rows)
print(f'{len(df)} files walked, {df.format.nunique()} distinct format labels')
df.head(12)

## 2. Format distribution

In [ ]:
vc = df['format'].value_counts()
fig, ax = plt.subplots(figsize=(11, max(3, 0.4*len(vc))))
vc[::-1].plot.barh(ax=ax); ax.set(title='files per format', xlabel='count')
plt.tight_layout(); plt.show()

## 3. Chunk-frequency census

How often each container chunk appears across the whole library — the metadata census.

In [ ]:
chunk_counter = Counter(cid.strip() for ids in df['chunk_ids'] for cid in ids)
if chunk_counter:
    cc = pd.Series(chunk_counter).sort_values()
    fig, ax = plt.subplots(figsize=(11, max(3, 0.35*len(cc))))
    cc.plot.barh(ax=ax); ax.set(title='chunk frequency across the library', xlabel='files containing it')
    plt.tight_layout(); plt.show()
else:
    print('no chunk-bearing (RIFF/AIFF-style) files in this set')

## 4. The oddballs (shiny candidates)

Files that carry a warning, or a **rare** chunk (seen in only 1–2 files). This is
automated shiny-hunting.

In [ ]:
rare = {c for c, n in chunk_counter.items() if n <= 2}
def oddness(r):
    ids = {c.strip() for c in r['chunk_ids']}
    return r['warnings'] + len(ids & rare)
df['oddness'] = df.apply(oddness, axis=1)
odd = df[df['oddness'] > 0].sort_values('oddness', ascending=False)
print(f'{len(odd)} oddballs (rare chunks: {sorted(rare)})')
odd[['file','format','warnings','chunk_ids','oddness']].head(15)

## 5. Chunk co-occurrence

Which chunks travel together (a fingerprint of the writing tool).

In [ ]:
all_chunks = sorted({c.strip() for ids in df['chunk_ids'] for c in ids})
if len(all_chunks) > 1:
    M = pd.DataFrame(0, index=all_chunks, columns=all_chunks)
    for ids in df['chunk_ids']:
        present = {c.strip() for c in ids}
        for a in present:
            for b in present:
                M.loc[a, b] += 1
    fig, ax = plt.subplots(figsize=(min(12, 1+0.6*len(all_chunks)),)*2)
    sns.heatmap(M, annot=True, fmt='d', cmap='mako', ax=ax)
    ax.set(title='chunk co-occurrence'); plt.tight_layout(); plt.show()
else:
    print('need >1 distinct chunk type for a co-occurrence matrix')

---
**Scale it:** set `LIBRARY = Path('~/sample_packs')` and re-run for a census over
1,286 real files — the format split, the vendor-chunk frequencies (LGWV/ResU/clm),
and the true oddballs jump right out.